# <center>YOLO Detection</center>

<center>
    <img width="100%" src="https://github.com/user-attachments/assets/a311a4ed-bbf2-43b5-8012-5f183a28a845" alt="YOLO11 performance plots">
</center>

## <center>YOLO 简史</center>

[YOLO](https://arxiv.org/abs/1506.02640) (You Only Look Once)，是一种流行的目标检测和图像分割模型，由华盛顿大学的Joseph Redmon和Ali Farhadi开发。自2015年推出以来，YOLO因其高速度和准确性迅速受到欢迎。

- 2016 年发布的 [YOLOv2](https://arxiv.org/abs/1612.08242) 通过引入批量归一化、锚框和尺寸聚类改进了原始模型。
- 2018 年推出的 [YOLOv3](https://pjreddie.com/media/files/papers/YOLOv3.pdf) 通过使用更高效的骨干网络、多锚和空间金字塔池进一步增强模型性能。
- 2020 年发布的 [YOLOv4](https://arxiv.org/abs/2004.10934) 引入了如Mosaic数据增强、新的无锚检测头和新的损失函数等创新。
- 2021 年推出的 [YOLOv5](https://github.com/ultralytics/yolov5) 进一步提高了模型性能，并增加了超参数优化、集成实验跟踪新功能。
- 2022 年由[美团](https://about.meituan.com/)开源的 [YOLOv6](https://github.com/meituan/YOLOv6)，被用于该公司的许多自动送货机器人中。
- 2022 年发布的 [YOLOv7](https://github.com/WongKinYiu/yolov7) 增加了额外的任务，如 COCO 关键点数据集的姿势估计。
- 2023 年由 [Ultralytics](https://www.ultralytics.com/) 发布的 [YOLOv8](https://github.com/ultralytics/ultralytics) 引入了新功能和改进，以增强性能、灵活性和效率并支持各种视觉AI任务。
- [YOLOv9](https://github.com/WongKinYiu/yolov9) 引入了可编程梯度信息 （PGI） 和广义高效层聚合网络 （GELAN） 等创新方法。
- [YOLOv10](https://github.com/THU-MIG/yolov10) 是由清华大学团队使用 Ultralytics 包创建，通过引入端到端头检测头，消除了非最大抑制，实现了实时目标检测的进步。
- [YOLO11](https://github.com/ultralytics/ultralytics) 是 Ultralytics 的最新YOLO模型在包括检测、分割、姿态估计、跟踪和分类在内的多项任务上提供最先进的（SOTA）性能，利用跨不同AI应用和领域的能力。

## <center>YOLO Detection 结构分析</center>
    
在上一章节中，我们了解到**将模型转换为 ONNX 格式，随后再转换为 TensorRT 引擎进行部署，这一流程因其高效性而成为目前最具性价比的解决方案。**在确保导出的 ONNX 模型中的算子得到 TensorRT 的支持后，我们只需掌握模型的输入和输出节点信息，便能够通过简单修改模板代码来快速完成部署工作。

In [ ]:
import netron

address = netron.start("./course_data/models/DET/yolov5n.onnx", verbosity=0, browse=False)
netron.widget(address)

In [ ]:
address = netron.start("./course_data/models/DET/yolov6n.onnx", verbosity=0, browse=False)
netron.widget(address)

In [ ]:
address = netron.start("./course_data/models/DET/yolov7t.onnx", verbosity=0, browse=False)
netron.widget(address)

In [ ]:
address = netron.start("./course_data/models/DET/yolov8n.onnx", verbosity=0, browse=False)
netron.widget(address)

In [ ]:
address = netron.start("./course_data/models/DET/yolov9t.onnx", verbosity=0, browse=False)
netron.widget(address)

In [ ]:
address = netron.start("./course_data/models/DET/yolov10n.onnx", verbosity=0, browse=False)
netron.widget(address)

In [ ]:
address = netron.start("./course_data/models/DET/yolo11n.onnx", verbosity=0, browse=False)
netron.widget(address)

**输入节点**：YOLO系列模型的输入节点具有一致性，它们都接受一个包含N个样本的批次，每个样本是一张具有3个颜色通道的高x宽像素的图像。

**输出节点**：输出结构各有不同，主要可以归纳为以下三种形式：

   - **[YOLOv5](https://github.com/ultralytics/yolov5/blob/master/models/yolo.py#L72)及其衍生变体（YOLOv5、YOLOv6、YOLOv7）**：输出维度为 `(batch, predicts, 5+classes)`，**其中 `5+classes` 表示 anchor 坐标 `(x, y, w, h)` 以及 anchor 的置信度，加上 classes 个类别的置信度。**

   - **[YOLOv8](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/nn/modules/head.py#L99)及其衍生变体（YOLOv8、YOLOv9、YOLO11）**：输出维度为 `(batch, 4+classes, predicts)`，**其中 `4+classes` 表示 anchor 坐标 `(x, y, w, h)`，加上 classes 个类别的置信度**。

   - 引入了NMS-Free机制的[YOLOv10](https://github.com/ultralytics/ultralytics/blob/main/ultralytics/nn/modules/head.py#L145)</b>：其输出维度为 `(batch, max_det, 6)`。**这里的 6 个通道分别代表：anchor 坐标 `(left, top, right, bottom)`，预测类别的置信度，以及预测类别的标签。**

## <center>注册 EfficientNMS 插件</center>

除了采用 NMS-Free 机制的 YOLOv10，其他 YOLO 变体在输出节点后都需要通过非极大值抑制（NMS）进行后处理。通过注册 NVIDIA TensorRT 提供的 [Efficient NMS Plugin](https://github.com/NVIDIA/TensorRT/tree/main/plugin/efficientNMSPlugin)，不仅可以加速 NMS 的处理过程，还能实现对不同 YOLO 变体输出节点的统一，从而简化部署流程。

目前，[YOLOv6](https://github.com/meituan/YOLOv6/tree/main/deploy/ONNX#tensorrt-backend-tensorrt-version-800)、[YOLOv7](https://github.com/WongKinYiu/yolov7#export) 和 [YOLOv9](https://github.com/WongKinYiu/yolov9/issues/130#issue-2162045461) 的官方仓库都支持导出集成了 EfficientNMS 插件的 ONNX 模型，以加速 TensorRT 的部署。下面，我们将以[YOLOv6的导出脚本](https://github.com/meituan/YOLOv6/blob/main/deploy/ONNX/export_onnx.py)为例，详细说明如何注册并导出带有 EfficientNMS 插件的 ONNX 模型。

In [ ]:
import onnx
import onnxsim
import torch
from ultralytics import YOLO
from course_functions.head import UltralyticsDetect

def export_model(model_path: str, output_path: str, batch: int = 1, 
                 imgsz: tuple = (640, 640), max_boxes: int = 100, 
                 iou_thres: float = 0.45, conf_thres: float = 0.25, 
                 opset_version: int = 11) -> None:
    """
    将 YOLO 模型导出为 ONNX 格式，并修改检测头。
    
    参数：
    - model_path: YOLO 模型权重文件的路径。
    - output_path: 导出的 ONNX 模型保存路径。
    - batch: 模型的批量大小，默认为 1。
    - imgsz: 模型输入的图像尺寸，默认为 (640, 640)。
    - max_boxes: 每张图片的最大检测数量，默认为 100。
    - iou_thres: NMS 的 IoU 阈值，默认为 0.25。
    - conf_thres: 检测的置信度阈值，默认为 0.25。
    - opset_version: ONNX opset 版本，默认为 11。
    """
    # 加载指定权重的 YOLO 模型并设置为 CPU 模式
    model = YOLO(model=model_path, verbose=False).model.to('cpu')
    
    # 修改 Detect 层的参数
    for m in model.modules():
        if m.__class__.__name__ == "Detect":
            m.__class__ = type("Detect", (UltralyticsDetect,), {
                "dynamic": batch <= 0,  # 是否需要动态批量大小
                "max_det": max_boxes,  # 每张图片的最大检测数量
                "iou_thres": iou_thres,  # NMS 的 IoU 阈值
                "conf_thres": conf_thres,  # 检测的置信度阈值
            })
            break

    # 创建模型的虚拟输入张量
    dummy_input = torch.randn(batch, 3, *imgsz).to('cpu')
    model.eval().float()  # 将模型设置为评估模式，并确保其为浮点精度
    model(dummy_input)  # 使用虚拟输入运行模型，以确保其正确配置

    # 将模型导出为 ONNX 格式
    torch.onnx.export(
        model,
        dummy_input,
        output_path,
        opset_version=opset_version,
        input_names=['images'],
        output_names=["num_dets", "det_boxes", "det_scores", "det_classes"],
        dynamic_axes={
            "images": {0: "batch", 2: "height", 3: "width"},
            "num_dets": {0: "batch"},
            "det_boxes": {0: "batch"},
            "det_scores": {0: "batch"},
            "det_classes": {0: "batch"},
        } if batch <= 0 else None,  # 如果批量大小是动态的，则定义动态轴
    )

    # 加载导出的 ONNX 模型并进行验证
    model_onnx = onnx.load(output_path)
    onnx.checker.check_model(model_onnx)

    # 则更新输出节点维度
    shapes = {
        'num_dets': ["batch" if batch <= 0 else batch, 1],
        'det_boxes': ["batch" if batch <= 0 else batch, max_boxes, 4],
        'det_scores': ["batch" if batch <= 0 else batch, max_boxes],
        'det_classes': ["batch" if batch <= 0 else batch, max_boxes],
    }
    for node in model_onnx.graph.output:
        for idx, dim in enumerate(node.type.tensor_type.shape.dim):
            dim.dim_param = str(shapes[node.name][idx])

    # 简化 ONNX 模型
    model_onnx, check = onnxsim.simplify(model_onnx)
    assert check, "Simplified ONNX model could not be validated"
    onnx.save(model_onnx, output_path)  # 保存简化后的 ONNX 模型

# 调用函数导出模型
export_model("./course_data/models/DET/yolo11n.pt", "./course_data/models/DET/yolo11n_with_plugin.onnx")

In [ ]:
address = netron.start("./course_data/models/DET/yolo11n_with_plugin.onnx", verbosity=0, browse=False)
netron.widget(address)

<b><font color="red">作业：</font></b>完成了对YOLOv6模型的讲解并实现了对 [Ultralytics-YOLO](https://github.com/ultralytics/ultralytics) 导出集成 EfficientNMS 插件的 ONNX 模型后，相信大家已经摩拳擦掌，准备亲自动手实践了。那么，接下来的挑战是：尝试为 [YOLOv5](https://github.com/ultralytics/yolov5) 导出集成 EfficientNMS 插件的 ONNX 模型。同时，也可以尝试对 [YOLOv10](https://github.com/THU-MIG/yolov10) 的输出节点进行调整，以确保它们与 EfficientNMS 插件的输出节点格式一致。

## <center>总结</center>

本节课程我们深入探讨了 YOLO 系列模型的发展历程及其结构特点。我们学习了如何将 YOLO 模型导出为 ONNX 格式，并注册 EfficientNMS 插件以加速 TensorRT 的部署过程。

通过实践，我们掌握了如何修改 YOLO 模型的检测头，并成功导出了集成 EfficientNMS 插件的 ONNX 模型。这不仅提高了模型的推理速度，还简化了部署流程。希望大家能够通过完成作业，进一步巩固所学知识，并在实际项目中灵活运用这些技能。